# F1 · Five implementations compared: speed and quality as the data grows

**Type**: Compare · **Level**: ●●○ independent · **Time**: about 50 minutes · **core path ★**

## Objective

Compare `GradientBoostingClassifier` (classic scikit-learn), `HistGradientBoostingClassifier`, `XGBClassifier` with `tree_method="hist"` and `"exact"` and `RandomForestClassifier` on three training set sizes (2000, 8000, 32000 examples): how much the time grows, what the AUC is worth and what matters more (algorithm or amount of data).

## Dataset

`make_classification(40000, 20 features, 8 informative, flip_y=0.05)`: the first `n` examples are the training set, the last 8000 the test set. All the models have 100 trees of depth 4 (the forest has free depth).

## Educational purpose

Choose an implementation with judgment: the classic scikit-learn boosting is linear in `n × features × thresholds` per tree and does not scale; the histogram versions (Hist, XGBoost `hist`) quantize the features into 256 bins and grow almost linearly; the quality changes little, the time enormously.

## Task

1. Build `table` (`pd.DataFrame` with a multi-index `(method, n)`, columns `seconds` (`fit` time with `time.perf_counter`) and `auc` on the test set) for the five methods and the three `n`.
2. Save `time_growth` (`pd.Series`, per method: `seconds(n=32000) / seconds(n=2000)`) and `gb_speedup` (time of `GradientBoostingClassifier` divided by the time of `XGBClassifier hist`, at `n = 32000`).
3. Save `auc_spread_32000` (maximum AUC minus minimum among the five methods at `n = 32000`) and `data_gain` (`pd.Series`, per method: `auc(32000) − auc(2000)`).
4. Save `scalability_conclusion` (`"classic gb does not scale"` or `"all scale equally"`) and `matters_more` (`"algorithm"` or `"data"`: what moves the AUC more, comparing `auc_spread_32000` with the mean of `data_gain`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import time

import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score

Xb, yb = make_classification(40000, n_features=20, n_informative=8, class_sep=0.7, flip_y=0.05, random_state=1)
X_te, y_te = Xb[-8000:], yb[-8000:]
sizes = [2000, 8000, 32000]
models = {
    "GradientBoosting": lambda: GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=0),
    "HistGradientBoosting": lambda: HistGradientBoostingClassifier(max_iter=100, max_depth=4, early_stopping=False, random_state=0),
    "XGBoost hist": lambda: xgb.XGBClassifier(n_estimators=100, max_depth=4, tree_method="hist", eval_metric="logloss", n_jobs=-1, random_state=0),
    "XGBoost exact": lambda: xgb.XGBClassifier(n_estimators=100, max_depth=4, tree_method="exact", eval_metric="logloss", n_jobs=-1, random_state=0),
    "RandomForest": lambda: RandomForestClassifier(100, n_jobs=-1, random_state=0),
}

## Your work

In [ ]:
index = pd.MultiIndex.from_product([list(models), sizes], names=["method", "n"])
table = pd.DataFrame(index=index, columns=["seconds", "auc"], dtype=float)
for name, create in models.items():
    for n in sizes:
        ...
table.round(3)

In [ ]:
time_growth = None
gb_speedup = None

In [ ]:
auc_spread_32000 = None
data_gain = None
scalability_conclusion = None
matters_more = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`t0 = time.perf_counter(); m = create().fit(X_tr, y_tr); seconds = time.perf_counter() - t0`, with `X_tr, y_tr = Xb[:n], yb[:n]`. Then `table.loc[(name, n)] = [seconds, roc_auc_score(y_te, m.predict_proba(X_te)[:, 1])]`. The series per method: `table["seconds"].unstack()`.'):
    check(table.notna().all().all() and table.shape == (15, 2) and (table["auc"] > 0.88).all(), "fifteen complete measurements, AUC always over 0.88", "time with time.perf_counter around fit")
    check(time_growth is not None and time_growth["GradientBoosting"] > 10 and time_growth["XGBoost hist"] < 6 and time_growth["HistGradientBoosting"] < 6, "the classic boosting grows by over 10 times from 2000 to 32000 examples, the histogram versions by less than 6", "seconds(32000) / seconds(2000)")
    check(gb_speedup is not None and gb_speedup > 15, "at 32000 examples the classic boosting is over 15 times slower than XGBoost hist", "GB time / XGBoost hist time")
    check(auc_spread_32000 is not None and auc_spread_32000 < 0.02 and (data_gain > 0.02).all(), "at 32000 examples the five methods differ by less than 2 points of AUC, while going from 2000 to 32000 examples gains over 2 points for each", "maximum AUC - minimum AUC; auc(32000) - auc(2000)")
    check(scalability_conclusion == "classic gb does not scale" and matters_more == "data", "the classic boosting does not scale and the factor that matters most is the amount of data", "compare 0.011 with 0.03")
done()

---
The commented solution is in `solutions/F_compare/F1_five_tree_implementations_compared.ipynb`. Open it after you pass the check, or after you have really tried.